# Qwen2.5-Coder-7B: Zero-shot Benchmark

A complete **no-training** benchmark on the repository's 269 Unity C# multiple-choice questions. The model is [Qwen/Qwen2.5-Coder-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct), matching the notebook link you shared.

## How to run
1. Upload **this notebook only** to [Google Colab](https://colab.research.google.com/): **File > Upload notebook**.
2. Select **Runtime > Change runtime type > T4 GPU**, or another available CUDA GPU.
3. Choose **Runtime > Run all**. Leave the settings below unchanged for your first full run.
4. Wait for all 269 questions to finish. A results ZIP downloads in the last cell.

**Everything is included:** test CSV, evaluation code, scoring, and export. No separate CSV upload, MCP connection, source repository, training notebook, or Hugging Face token is needed for the public default model. The first run downloads its model weights and needs internet access.

This evaluates a pretrained instruction model; it does not update weights or read `TrainData.csv`. Four-bit quantization reduces inference memory. The original assessment notebook and fixed `test_model` function are unchanged.


## 1. Install inference dependencies

Use the Colab runtime's existing PyTorch/CUDA installation. This cell installs Transformers 4.x, Accelerate for device placement, bitsandbytes for four-bit inference, and SentencePiece. Exact installed versions are saved with the results.


In [1]:
%pip install -q "transformers>=4.51,<5" accelerate bitsandbytes sentencepiece


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Check the GPU

If this cell reports no GPU, change the runtime to a GPU and rerun. No training will occur.


In [2]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then rerun."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GiB")
print(f"PyTorch: {torch.__version__}")


GPU: Tesla T4
GPU memory: 14.6 GiB
PyTorch: 2.14.1+cu130


## 3. Restore the included test data

The cell contains an encoded copy of the original test CSV. Encoding is only for packaging, not encryption. A SHA-256 check verifies the data. Existing files with different contents are preserved and trigger an error.

The model receives **only each question and its four answer choices**. The C# class metadata and answer labels are never added to its prompt. Labels are used only for scoring.


In [3]:
# Self-contained data setup: works in VS Code and in the Colab browser.
# Test labels are used for scoring only, never included in model prompts.
import base64
import csv
import hashlib
from pathlib import Path

TEST_DATA_BASE64 = (
    '...'
    'cyBoaWdoIHNjb3JlLixjDQo='
)
EXPECTED_SHA256 = "5e10ec19d61624ee3554814337cf961fe499f4a666881c866c172fa1e51454a2"
data_path = Path("TestData.csv")
embedded_data = base64.b64decode(TEST_DATA_BASE64)
assert hashlib.sha256(embedded_data).hexdigest() == EXPECTED_SHA256
if not data_path.exists():
    with data_path.open("xb") as handle:
        handle.write(embedded_data)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == EXPECTED_SHA256, (
    "Existing TestData.csv differs from the bundled dataset. "
    "Rename that file, then rerun this cell to restore the bundled version."
)
with data_path.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    required = {"class", "Question", "a", "b", "c", "d", "Answer"}
    assert required.issubset(reader.fieldnames or []), "Unexpected CSV columns"
    row_count = sum(1 for _ in reader)
assert row_count == 269, f"Expected 269 test questions, found {row_count}"
print(f"Ready: {row_count} test questions. No upload needed.")


Ready: 269 test questions. No upload needed.


## 4. Benchmark settings

Defaults evaluate all 269 questions, one at a time, using greedy decoding and a maximum of 128 generated tokens. There are no examples, retrieval, retries, or training. Invalid/missing final answers count as incorrect. Responses that reach the token budget are flagged.

For a quick execution check only, change `LIMIT` to `5`. Set it back to `None` for the full benchmark. Do not use test answers to tune prompts or generation settings.


In [4]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
FOUR_BIT = True
LIMIT = None  # Full benchmark. Use 5 only for a quick execution check.
MAX_NEW_TOKENS = 128
SEED = 42


## 5. Write the included evaluator

The evaluator uses the model's own chat template and decodes only newly generated tokens. It saves each response immediately, plus final accuracy, constant-choice reference scores, invalid-answer counts, timing, dataset hash, model revision, GPU, and package versions. If interrupted, completed predictions remain on the runtime; a final summary is produced only after successful completion.


In [5]:
%%writefile qwen_coder_benchmark.py
"""Zero-shot MCQ baseline. No training or training-data access."""

import argparse
import csv
import hashlib
import importlib.metadata
import json
from pathlib import Path
import re
import time


def build_prompt(row):
    choices = "\n".join(f"{key}) {row[key]}" for key in "abcd")
    return (
        "Answer the following multiple-choice question about a Unity C# game project.\n"
        "Choose the single best answer. End your response with Answer:a, Answer:b, "
        "Answer:c, or Answer:d.\n\n"
        f"{row['Question']}\n{choices}"
    )


def parse_answer(response):
    # An unfinished thinking block is not a final answer.
    if "<think>" in response and "</think>" not in response:
        return None
    final = response.rsplit("</think>", 1)[-1].strip()
    match = re.search(r"Answer\s*:\s*([abcd])\s*[.!]?\s*$", final, re.I)
    if match:
        return match.group(1).lower()
    return final.lower() if re.fullmatch(r"[abcd]", final, re.I) else None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", default="Qwen/Qwen2.5-Coder-7B-Instruct")
    parser.add_argument("--data", type=Path, default=Path("TestData.csv"))
    parser.add_argument("--output", type=Path, required=True,
                        help="New directory for this run; existing directories are rejected.")
    parser.add_argument("--limit", type=int, help="First N rows for smoke checks only")
    parser.add_argument("--thinking", action="store_true", help="Qwen3 thinking mode")
    parser.add_argument("--max-new-tokens", type=int)
    parser.add_argument("--four-bit", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(argv)
    if args.limit is not None and args.limit < 1:
        parser.error("--limit must be positive")
    if args.max_new_tokens is not None and args.max_new_tokens < 1:
        parser.error("--max-new-tokens must be positive")
    if args.thinking and not args.model.startswith("Qwen/Qwen3-"):
        parser.error("--thinking is implemented only for Qwen3 models")
    with args.data.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"class", "Question", "a", "b", "c", "d", "Answer"}
        if not required.issubset(reader.fieldnames or []):
            parser.error("Unexpected test CSV schema")
        rows = list(reader)
    if args.limit:
        rows = rows[:args.limit]
    if not rows or any(row["Answer"].strip().lower() not in "abcd" or
                       len(row["Answer"].strip()) != 1 for row in rows):
        parser.error("Empty dataset or invalid answer labels")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select a GPU runtime in Colab/Kaggle.")
    args.output.mkdir(parents=True, exist_ok=False)
    set_seed(args.seed)
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    config = {"device_map": "auto", "torch_dtype": torch.float16}
    if args.four_bit:
        config["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(args.model, **config).eval()
    max_tokens = args.max_new_tokens or (2048 if args.thinking else 128)
    generation = {"max_new_tokens": max_tokens, "do_sample": args.thinking,
                  "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}
    if args.thinking:
        generation.update(temperature=0.6, top_p=0.95, top_k=20)
    metadata = {
        "model": args.model, "model_commit": getattr(model.config, "_commit_hash", None),
        "dataset_sha256": hashlib.sha256(args.data.read_bytes()).hexdigest(),
        "rows": len(rows), "smoke_test": args.limit is not None, "seed": args.seed,
        "thinking": args.thinking, "four_bit": args.four_bit, "generation": generation,
        "gpu": torch.cuda.get_device_name(0), "prompt_example": build_prompt(rows[0]),
        "versions": {p: importlib.metadata.version(p) for p in
                     ["torch", "transformers", "accelerate", "bitsandbytes"]},
        "protocol": "Zero-shot; no training examples or source code; invalid answers count as wrong.",
    }
    (args.output / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    predictions = []
    start = time.perf_counter()
    with (args.output / "predictions.jsonl").open("x", encoding="utf-8") as handle:
        for index, row in enumerate(rows):
            messages = [{"role": "user", "content": build_prompt(row)}]
            template_args = {}
            if args.model.startswith("Qwen/Qwen3-"):
                template_args["enable_thinking"] = args.thinking
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, **template_args)
            inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
            tick = time.perf_counter()
            with torch.inference_mode():
                output = model.generate(**inputs, **generation)
            generated = output[0, inputs.input_ids.shape[1]:]
            response = tokenizer.decode(generated, skip_special_tokens=True)
            answer = parse_answer(response)
            result = {
                "row": index, "class": row["class"], "question": row["Question"],
                "response": response, "prediction": answer,
                "truth": row["Answer"].strip().lower(),
                "correct": answer == row["Answer"].strip().lower(),
                "seconds": time.perf_counter() - tick,
                "generated_tokens": len(generated),
                "token_budget_reached": len(generated) >= max_tokens,
            }
            predictions.append(result)
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
            handle.flush()
            print(f"{index + 1}/{len(rows)} correct={sum(p['correct'] for p in predictions)}", flush=True)
    correct = sum(p["correct"] for p in predictions)
    labels = {key: sum(p["truth"] == key for p in predictions) for key in "abcd"}
    summary = {
        "model": args.model, "questions": len(rows), "correct": correct,
        "accuracy": correct / len(rows),
        "invalid_answers": sum(p["prediction"] is None for p in predictions),
        "token_budget_reached": sum(p["token_budget_reached"] for p in predictions),
        "uniform_random_expected_accuracy": 0.25,
        "label_counts": labels, "constant_choice_accuracy": {k: v / len(rows) for k, v in labels.items()},
        "elapsed_seconds": time.perf_counter() - start,
        "smoke_test": args.limit is not None,
    }
    (args.output / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


Writing qwen_coder_benchmark.py


## 6. Run all questions

The initial model download and loading may take several minutes. Once evaluation starts, the output shows progress as `1/269`, `2/269`, and so on. Each run uses a new result directory. Running this cell again repeats inference, without training.


In [6]:
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = f"results/qwen25-coder-7b-zero-shot-{stamp}"
command = [sys.executable, "-u", "qwen_coder_benchmark.py", "--model", MODEL,
           "--data", "TestData.csv", "--output", RUN_DIR,
           "--seed", str(SEED), "--max-new-tokens", str(MAX_NEW_TOKENS)]
if FOUR_BIT:
    command.append("--four-bit")
if LIMIT is not None:
    command.extend(["--limit", str(LIMIT)])
print(f"Model: {MODEL} | Training: none | Results: {RUN_DIR}", flush=True)
print("Downloading/loading the model first; question progress follows.", flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f"Benchmark failed with exit code {exit_code}. Read the error above.")


Model: Qwen/Qwen2.5-Coder-7B-Instruct | Training: none | Results: results/qwen25-coder-7b-zero-shot-20261113T042115231370Z
Downloading/loading the model first; question progress follows.
I0000 00:00:1794543694.177834     869 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794543699.441621     869 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 4 files: 100%|██████████| 4/4 [03:11<00:00, 47.94s/it] 

Loading checkpoint shards: 100%|██████████| 4/4 [01:07<00:00, 16.97s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
1/269 correct=1
2/269 correct=2
3/269 correc

## 7. View results

Uniform random guessing has **25% expected accuracy**. Constant-choice accuracies show what always selecting a particular letter would score on this dataset. They are descriptive references, not model results. Invalid answers count as wrong in the main accuracy.

This benchmark measures existing task performance, not general reasoning in isolation. Naming clues can make some questions guessable; implementation details may require project knowledge the model has never seen. This is a four-bit inference baseline, so record that setting in comparisons. Use training-derived validation for future development and disclose reuse of this test set.


In [7]:
import json
from IPython.display import Markdown, display

summary = json.loads((Path(RUN_DIR) / "summary.json").read_text())
display(Markdown(
    f"### {'Smoke check only' if summary['smoke_test'] else 'Full benchmark results'}\n\n"
    f"| Metric | Value |\n|---|---|\n"
    f"| Model | {summary['model']} |\n"
    f"| Accuracy | {summary['accuracy']:.2%} |\n"
    f"| Correct / total | {summary['correct']} / {summary['questions']} |\n"
    f"| Invalid answers | {summary['invalid_answers']} |\n"
    f"| Token budget reached | {summary['token_budget_reached']} |\n"
    f"| Random-choice expectation | 25% |\n"
    f"| Inference time | {summary['elapsed_seconds'] / 60:.1f} minutes |"
))
print("Constant-choice accuracies:")
for answer, accuracy in summary["constant_choice_accuracy"].items():
    print(f"  Always choose {answer}: {accuracy:.2%}")


### Full benchmark results

| Metric | Value |
|---|---|
| Model | Qwen/Qwen2.5-Coder-7B-Instruct |
| Accuracy | 67.66% |
| Correct / total | 182 / 269 |
| Invalid answers | 24 |
| Token budget reached | 2 |
| Random-choice expectation | 25% |
| Inference time | 2.0 minutes |

Constant-choice accuracies:
  Always choose a: 36.06%
  Always choose b: 30.48%
  Always choose c: 18.22%
  Always choose d: 15.24%


## 8. Download results

The ZIP contains `summary.json`, `metadata.json`, raw per-question `predictions.jsonl`, a convenient `predictions.csv`, and the evaluator source. Keep this archive for the later fine-tuning comparison. This download cell is intended for the **Colab website**.


In [8]:
import csv
import shutil
from google.colab import files

result_path = Path(RUN_DIR)
predictions = [json.loads(line) for line in (result_path / "predictions.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
with (result_path / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))
    writer.writeheader()
    writer.writerows(predictions)
shutil.copyfile("qwen_coder_benchmark.py", result_path / "qwen_coder_benchmark.py")
archive = shutil.make_archive(str(result_path), "zip", result_path)
print(f"Saved results: {archive}")
files.download(archive)


Saved results: /content/results/qwen25-coder-7b-zero-shot-20261113T042115231370Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>